# Reinforcement Learning Track · Proximal Policy Optimization

**Track:** Reinforcement Learning  
**Objective:** Master the principles and applications of Proximal Policy Optimization.

---



### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** PPO's genius is making policy gradient methods PRACTICAL. Before PPO, policy gradient methods were fragile — one bad update could destroy the policy. PPO adds a safety mechanism (the clip) that prevents catastrophic policy changes while still allowing efficient learning from batched data.

**Mental Model:** Imagine adjusting a satellite dish to get better TV signal. REINFORCE moves the dish once per signal test. PPO takes multiple readings, then makes several small adjustments — but with a safety lock that prevents turning the dish more than 20% from its current position. This way you improve faster without losing the signal entirely.

**Common Junior Pitfall:** Setting the clip range too large (ε > 0.3). This defeats PPO's purpose — the policy can change too much per update, leading to instability. The standard ε=0.2 was tuned extensively and rarely needs changing.

---

---
## 5 · Bridge to LLMs (RLHF)

You have just implemented the exact algorithm used to align models like ChatGPT, Claude, and Llama 3.

### How PPO Aligns Language Models

In standard RL (like the CartPole task above):
- **State:** Cart position, pole angle
- **Action:** Move left or right
- **Reward:** +1 for every step the pole stays upright

In **Reinforcement Learning from Human Feedback (RLHF)**:
- **State:** The conversational prompt (e.g. "Write a polite email")
- **Action:** The generated response token by token
- **Reward (The 'RM'):** There is no hardcoded physics engine. Instead, a separate **Reward Model** (a neural network trained on human preferences) dynamically outputs a scalar $+1$ or $-1$ evaluating the politeness of your generated response.

PPO pushes the LLM to generate responses that yield high scores from the Reward Model, while using the KL-divergence penalty constraint to prevent the model from generating absolute garbage that nominally games the reward system.

*(This bridges directly into the main AI Engineering track. For deep implementation of the RLHF Instruct pipeline, see **Main Track Notebook `17_02_rlhf_alignment.ipynb`**).*


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Trust Region Problem](#1-the-trust-region-problem)
  * [Why We Need PPO](#why-we-need-ppo)
  * [The Solution: Trust Regions](#the-solution-trust-regions)
* [2 · PPO-Clip: The Core Mechanism](#2-ppo-clip-the-core-mechanism)
  * [The Probability Ratio](#the-probability-ratio)
  * [The Clipped Objective](#the-clipped-objective)
* [3 · GAE: Generalized Advantage Estimation](#3-gae-generalized-advantage-estimation)
* [4 · Industry-Standard Enhancements](#4-industry-standard-enhancements)
  * [1. Vectorized Environments (Parallelism)](#1-vectorized-environments-parallelism)
  * [2. Orthogonal Initialization](#2-orthogonal-initialization)
  * [3. Learning Rate Annealing](#3-learning-rate-annealing)
* [5 · Bridge to LLMs (RLHF)](#5-bridge-to-llms-rlhf)
  * [How PPO Aligns Language Models](#how-ppo-aligns-language-models)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Clip Ablation](#exercise-1-clip-ablation)
  * [Exercise 2: Continuous Actions Challenge](#exercise-2-continuous-actions-challenge)
  * [Exercise 3: LunarLander Integration](#exercise-3-lunarlander-integration)


In [ ]:
!pip install -q torch numpy matplotlib gymnasium

## 1 · The Trust Region Problem

### Why We Need PPO

**Problem with REINFORCE/Actor-Critic:** One gradient step. Collect data → compute gradient → update → throw away data. Wasteful!

**Naive fix:** Update multiple times on the same data.

**Why this breaks:** After the first update, the policy has CHANGED. Data was collected under the OLD policy. Using old data with the new policy creates a mismatch. Too many updates → policy diverges.

### The Solution: Trust Regions

Allow multiple updates, but CONSTRAIN how much the policy can change:

| Method | How It Constrains | Complexity |
|--------|------------------|------------|
| **TRPO** (2015) | KL-divergence constraint | Complex (requires conjugate gradient) |
| **PPO-Clip** (2017) | Clips probability ratio | Simple (just a min/clip) |
| **PPO-Penalty** | KL penalty term | Medium |

PPO-Clip won because it's the simplest and works just as well.

---
## 2 · PPO-Clip: The Core Mechanism

### The Probability Ratio

$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$$

- $r_t = 1$: new policy same as old → no change
- $r_t > 1$: new policy makes this action MORE likely
- $r_t < 1$: new policy makes this action LESS likely

### The Clipped Objective

$$L^{\text{CLIP}}(\theta) = \mathbb{E} \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

In plain English with ε=0.2:
- If the action was GOOD ($\hat{A} > 0$): allow increasing its probability, but cap the increase at 1.2× the old probability
- If the action was BAD ($\hat{A} < 0$): allow decreasing its probability, but cap the decrease at 0.8× the old probability
- Result: the policy changes gradually, never making drastic jumps

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Visualize the PPO clip mechanism
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epsilon = 0.2
ratios = np.linspace(0.5, 1.8, 200)

for ax, advantage, title in [(axes[0], 1.0, 'Good Action (A > 0)'),
                              (axes[1], -1.0, 'Bad Action (A < 0)')]:
    # Unclipped objective
    unclipped = ratios * advantage
    # Clipped objective
    clipped_ratios = np.clip(ratios, 1 - epsilon, 1 + epsilon)
    clipped = clipped_ratios * advantage
    # PPO objective: min of both
    ppo_obj = np.minimum(unclipped, clipped) if advantage > 0 else np.maximum(unclipped, clipped)
    # For A > 0, min clips the upside. For A < 0, we want max of negative values (less negative)
    ppo_obj = np.where(advantage > 0, np.minimum(unclipped, clipped), np.maximum(unclipped, clipped))
    
    ax.plot(ratios, unclipped, label='Unclipped', lw=2, linestyle='--', alpha=0.5)
    ax.plot(ratios, ppo_obj, label='PPO-Clip', lw=3, color='steelblue')
    ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=1-epsilon, color='red', linestyle='--', alpha=0.5, label=f'1-ε={1-epsilon}')
    ax.axvline(x=1+epsilon, color='red', linestyle='--', alpha=0.5, label=f'1+ε={1+epsilon}')
    ax.set_xlabel('Probability Ratio r(θ)')
    ax.set_ylabel('Objective')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('PPO Clipping: Prevents Policy from Changing Too Much', fontweight='bold')
plt.tight_layout()
plt.show()

print('Left plot (good action, A>0):')
print('  Without clip: ratio can go to 2x, 5x — dangerously large update')
print('  With clip: ratio capped at 1.2 — controlled improvement')
print()
print('Right plot (bad action, A<0):')
print('  Without clip: ratio can go to 0 — might remove action entirely')
print('  With clip: ratio capped at 0.8 — gradual reduction')

---
## 3 · GAE: Generalized Advantage Estimation

The advantage $A_t$ in PPO uses **GAE** (Generalized Advantage Estimation) for the best bias-variance tradeoff:

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{T-t} (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error.

| λ value | Behavior | Bias | Variance |
|---------|----------|------|----------|
| λ=0 | One-step TD (pure critic) | High | Low |
| λ=1 | Monte Carlo (full returns) | Low | High |
| **λ=0.95** | **Best tradeoff (default)** | **Medium** | **Medium** |

Think of λ as controlling how far into the future the advantage calculation looks.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation (GAE).
    
    This is the advantage estimator used in PPO and RLHF.
    
    Args:
        rewards: list of rewards [r_0, r_1, ..., r_T]
        values:  list of V(s) estimates [V(s_0), ..., V(s_T), V(s_{T+1})]
        dones:   list of done flags
        gamma:   discount factor
        lam:     GAE lambda (bias-variance tradeoff)
    
    Returns:
        advantages: GAE advantage estimates
        returns:    targets for value function training
    """
    T = len(rewards)
    advantages = np.zeros(T)
    gae = 0
    
    for t in reversed(range(T)):
        next_value = values[t + 1] if t + 1 < len(values) else 0
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages[t] = gae
    
    returns = advantages + np.array(values[:T])  # V(s) + A(s,a) = Q(s,a) ≈ return
    return advantages, returns

# Demo
rewards = [1, 1, 1, 1, 10]  # Big reward at the end
values =  [2, 3, 4, 5, 8, 0]  # V(s) estimates (last is terminal)
dones =   [0, 0, 0, 0, 1]

advantages, returns = compute_gae(rewards, values, dones)
print('GAE Computation:')
print(f'  Rewards:    {rewards}')
print(f'  Values:     {values[:-1]}')
print(f'  Advantages: {advantages.round(3).tolist()}')
print(f'  Returns:    {returns.round(3).tolist()}')
print(f'\n  Last step has highest advantage — the big reward was unexpected (10 vs V=8).')
print(f'  GAE propagates this "surprisingly good" signal backward through time.')

---
## 4 · Industry-Standard Enhancements

Before writing the final training loop, production-grade PPO heavily relies on three critical engineering optimizations. Without these, PPO often fails or learns extremely slowly on continuous tasks like `LunarLander-v3`.

### 1. Vectorized Environments (Parallelism)
Instead of stepping through one environment, we run $N$ environments simultaneously (e.g., `gymnasium.vector`).
- **Speed:** Exploits batch processing on CPUs/GPUs.
- **Stability:** Breaking correlated data. Looking at $N$ different environment states at time $t$ provides a much richer, diverse gradient signal than looking at 1 state sequence.

### 2. Orthogonal Initialization
Normal random weight initialization often causes exploding or vanishing gradients in deep RL. **Orthogonal Initialization** forces the weight matrices to be mathematically orthogonal. 
- It guarantees that passing inputs through the layer preserves their variance.
- Critical for PPO stability, as preserving the exact feature geometry across the Actor and Critic heads prevents massive early policy collapse.

### 3. Learning Rate Annealing
RL data is extremely noisy. As the agent gets closer to the optimal policy, keeping the learning rate static causes it to wildly "overshoot" the local minimum. Linearly decaying the LR to exactly 0 by the end of training forces the policy to smoothly settle into optimum.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """Orthogonal initialization function."""
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class PPOActorCritic(nn.Module):
    """Robust Vectorized Actor-Critic for Discrete Action Spaces."""
    def __init__(self, state_dim, action_dim):
        super().__init__()
        # Critic network
        self.critic = nn.Sequential(
            layer_init(nn.Linear(state_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0), # std=1 for value
        )
        # Actor network
        self.actor = nn.Sequential(
            layer_init(nn.Linear(state_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, action_dim), std=0.01), # std=0.01 forces initial probabilities near uniform
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

def ppo_train_vectorized():
    # Hyperparameters
    num_envs = 4
    num_steps = 128         # steps per env before update
    num_updates = 150       # total updates
    lr = 5e-4
    gamma = 0.99
    gae_lambda = 0.95
    clip_coef = 0.2
    ent_coef = 0.01
    update_epochs = 4
    batch_size = int(num_envs * num_steps)
    
    # Vector Environment Setup
    envs = gym.make_vec('CartPole-v1', num_envs=num_envs)
    agent = PPOActorCritic(envs.single_observation_space.shape[0], envs.single_action_space.n)
    optimizer = optim.Adam(agent.parameters(), lr=lr, eps=1e-5)
    
    # Storage setup
    obs = torch.zeros((num_steps, num_envs) + envs.single_observation_space.shape)
    actions = torch.zeros((num_steps, num_envs) + envs.single_action_space.shape)
    logprobs = torch.zeros((num_steps, num_envs))
    rewards = torch.zeros((num_steps, num_envs))
    dones = torch.zeros((num_steps, num_envs))
    values = torch.zeros((num_steps, num_envs))
    
    next_obs, _ = envs.reset()
    next_obs = torch.Tensor(next_obs)
    next_done = torch.zeros(num_envs)
    
    avg_return_history = []
    
    for update in range(1, num_updates + 1):
        # Learning Rate Annealing
        frac = 1.0 - (update - 1.0) / num_updates
        lrnow = frac * lr
        optimizer.param_groups[0]["lr"] = lrnow
        
        # --- Phase 1: Collect Vectorized Rollouts ---
        for step in range(0, num_steps):
            obs[step] = next_obs
            dones[step] = next_done
            
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob
            
            # Step the environments in parallel!
            next_obs_np, reward, terminated, truncated, infos = envs.step(action.cpu().numpy())
            rewards[step] = torch.tensor(reward).view(-1)
            next_obs, next_done = torch.Tensor(next_obs_np), torch.Tensor(terminated | truncated)
            
        # Estimate performance
        avg_return_history.append(rewards.sum(0).mean().item())
        
        # --- Phase 2: Compute GAE ---
        with torch.no_grad():
            next_value = agent.get_value(next_obs).reshape(1, -1)
            advantages = torch.zeros_like(rewards)
            lastgaelam = 0
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    nextnonterminal = 1.0 - next_done
                    nextvalues = next_value
                else:
                    nextnonterminal = 1.0 - dones[t + 1]
                    nextvalues = values[t + 1]
                delta = rewards[t] + gamma * nextvalues * nextnonterminal - values[t]
                advantages[t] = lastgaelam = delta + gamma * gae_lambda * nextnonterminal * lastgaelam
            returns = advantages + values
            
        # Flatten the batch
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)
        
        # --- Phase 3: Optimize PPO Objectively Multiple Epochs ---
        b_inds = np.arange(batch_size)
        for epoch in range(update_epochs):
            np.random.shuffle(b_inds)
            _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[b_inds], b_actions.long()[b_inds])
            logratio = newlogprob - b_logprobs[b_inds]
            ratio = logratio.exp()
            
            mb_advantages = b_advantages[b_inds]
            mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
            
            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - clip_coef, 1 + clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()
            
            v_loss = 0.5 * ((newvalue.view(-1) - b_returns[b_inds]) ** 2).mean()
            entropy_loss = entropy.mean()
            
            loss = pg_loss - ent_coef * entropy_loss + v_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
            optimizer.step()

    envs.close()
    return avg_return_history

print('Training Vectorized PPO on CartPole-v1...')
ppo_history = ppo_train_vectorized()

# Plot results
plt.figure(figsize=(10, 5))
window = 5
smoothed = [np.mean(ppo_history[max(0,i-window):i+1]) for i in range(len(ppo_history))]
plt.plot(smoothed, lw=2, color='steelblue')
plt.xlabel('Algorithm Update Index')
plt.ylabel('Score per Batch (smoothed)')
plt.title('PPO Vectorized Training: CartPole-v1')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ppo_cartpole.png', dpi=100)
plt.show()

print(f'Done! PPO rapidly learns and stabilizes the physical pole balancing logic.')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class PPOActorCritic(nn.Module):
    """Actor-Critic network for PPO."""
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, action_dim), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    
    def forward(self, state):
        return self.actor(state), self.critic(state)
    
    def act(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0)
        probs, value = self.forward(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action).item(), value.item()

class SimpleBalanceEnv:
    def __init__(self):
        self.reset()
    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, size=4)
        self.steps = 0
        return self.state.copy()
    def step(self, action):
        force = 1.0 if action == 1 else -1.0
        self.state[3] += force * 0.02 + np.random.normal(0, 0.01)
        self.state[2] += self.state[3] * 0.02
        self.state[1] += force * 0.01
        self.state[0] += self.state[1] * 0.02
        self.steps += 1
        done = abs(self.state[2]) > 0.5 or abs(self.state[0]) > 2.0 or self.steps >= 200
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

def ppo_train(env, total_episodes=500, gamma=0.99, lam=0.95,
              clip_epsilon=0.2, epochs_per_update=4, lr=3e-4):
    """PPO Training Loop.
    
    The key difference from REINFORCE:
    1. Collect a batch of episodes
    2. Compute advantages with GAE
    3. Update policy MULTIPLE TIMES on the same data (with clipping)
    """
    model = PPOActorCritic(state_dim=4, action_dim=2)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    all_rewards = []
    
    for episode in range(total_episodes):
        # --- Step 1: Collect trajectory ---
        states, actions, rewards, log_probs_old, values, dones = [], [], [], [], [], []
        state = env.reset()
        
        while True:
            action, log_prob, value = model.act(state)
            next_state, reward, done = env.step(action)
            
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            log_probs_old.append(log_prob)
            values.append(value)
            dones.append(done)
            
            state = next_state
            if done:
                break
        
        all_rewards.append(sum(rewards))
        
        # --- Step 2: Compute GAE advantages ---
        values.append(0)  # Terminal value
        advantages, returns = compute_gae(rewards, values, dones, gamma, lam)
        
        # Convert to tensors
        states_t = torch.FloatTensor(np.array(states))
        actions_t = torch.LongTensor(actions)
        old_log_probs_t = torch.FloatTensor(log_probs_old)
        advantages_t = torch.FloatTensor(advantages)
        returns_t = torch.FloatTensor(returns)
        
        # Normalize advantages
        if len(advantages_t) > 1:
            advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        
        # --- Step 3: PPO update (MULTIPLE epochs on same data!) ---
        for _ in range(epochs_per_update):
            probs, values_pred = model(states_t)
            dist = torch.distributions.Categorical(probs)
            new_log_probs = dist.log_prob(actions_t)
            entropy = dist.entropy().mean()
            
            # Probability ratio: π_new / π_old
            ratio = torch.exp(new_log_probs - old_log_probs_t)
            
            # PPO Clipped objective
            surr1 = ratio * advantages_t
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * advantages_t
            actor_loss = -torch.min(surr1, surr2).mean()
            
            # Critic loss: MSE between predicted and actual returns
            critic_loss = nn.functional.mse_loss(values_pred.squeeze(), returns_t)
            
            # Total loss = actor + critic - entropy bonus
            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
    
    return model, all_rewards

# Train PPO
env = SimpleBalanceEnv()
print('Training PPO...')
ppo_model, ppo_rewards = ppo_train(env, total_episodes=400)

# Plot results
window = 30
smoothed = [np.mean(ppo_rewards[max(0,i-window):i+1]) for i in range(len(ppo_rewards))]

plt.figure(figsize=(10, 5))
plt.plot(smoothed, lw=2, color='steelblue', label='PPO')
plt.axhline(y=200, color='green', linestyle='--', alpha=0.7, label='Maximum (200)')
plt.xlabel('Episode')
plt.ylabel('Total Reward (smoothed)')
plt.title('PPO Training: Balance Task')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Early: {np.mean(ppo_rewards[:30]):.0f} steps | Late: {np.mean(ppo_rewards[-30:]):.0f} steps')
print(f'PPO learned to balance! This is the SAME algorithm that fine-tunes ChatGPT.')

---

## 🔨 Practical Practice

### Exercise 1: Clip Ablation
Train PPO with ε = 0.05, 0.1, 0.2, 0.3, 0.5. Compare learning curves. Which is most stable? Which is fastest?

### Exercise 2: Continuous Actions Challenge
Modify our `PPOActorCritic` for continuous action spaces. The actor should output `(mean, log_std)` of a Gaussian, and you sample actions from `N(mean, std)`. Use this for `Pendulum-v1`.

### Exercise 3: LunarLander Integration
Change `gym.make_vec('CartPole-v1')` to `LunarLander-v3`. Ensure you have `swig` and `box2d` Python dependencies installed locally, and watch the generalized PPO logic solve complex multi-dimensional thrust optimizations instantly!

---

**Next →** RL 06: RLHF — From PPO to Language Model Alignment
